# StyleMatch hubness-aware reranking

This notebook reuses the frozen expanded-index score matrices. It does not train or encode text. The comparison is exploratory source-grouped cross-fitting; deployment still requires a newly frozen confirmatory evaluation. Research basis: `docs/hubness_reranking_research_2026.md`.

In [ ]:
from google.colab import drive
from pathlib import Path
import json, os, subprocess, sys, zipfile
import pandas as pd

drive.mount('/content/drive')
REPO = Path('/content/drive/MyDrive/style_matching')
os.chdir(REPO)
GUT = REPO / 'artifacts/source_expansion_v2/gutenberg_targeted_v1'
SPLIT = GUT / 'source_heldout_splits.parquet'
SCORES = GUT / 'frozen_encoder_eval/style_embedding_scores.npz'
INDEX = GUT / 'index'
OUT = GUT / 'hubness_reranking_v2'
for path in (SPLIT, SCORES, INDEX / 'metadata.json'):
    assert path.exists(), f'Missing completed Part 8 artifact: {path}'

def run(cmd):
    print('>>>', ' '.join(map(str, cmd)), flush=True)
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='', flush=True)
    code = process.wait()
    if code:
        raise RuntimeError(f'command failed with exit {code}: {" ".join(map(str, cmd))}')


## 1 · Create the expanded deployment-index ZIP

The ZIP is written directly to `My Drive/style_matching/stylematch_index_gutenberg_v3.zip`. Large build caches are excluded; runtime files are retained.

In [ ]:
INDEX_ZIP = REPO / 'stylematch_index_gutenberg_v3.zip'
excluded = {'chunk_embeddings.npz', 'topic_chunk_embeddings.npz'}
with zipfile.ZipFile(INDEX_ZIP, 'w', zipfile.ZIP_DEFLATED) as archive:
    for path in INDEX.rglob('*'):
        if path.is_file() and path.name not in excluded:
            archive.write(path, path.relative_to(INDEX))
print('ZIP:', INDEX_ZIP)
print('SIZE MB:', round(INDEX_ZIP.stat().st_size / 1024**2, 1))

## 2 · Cross-fitted concentration tests

Candidate-specific empirical nulls, mutual-proximity-style calibration, multi-view rank fusion, and a fine local-density grid are compared against the uncorrected centroid.

In [ ]:
run([sys.executable, 'scripts/evaluate_hubness_reranking.py',
     '--input', str(SPLIT), '--scores', str(SCORES),
     '--output-dir', str(OUT), '--folds', '5', '--top-k', '10',
     '--bootstrap-runs', '5000', '--seed', '20260902'])

## 3 · Decision table and concentrated authors

In [ ]:
report = json.loads((OUT / 'hubness_reranking_metrics.json').read_text())
rows = []
for name, result in report['methods'].items():
    rows.append({
        'method': name,
        'MRR': result['metrics']['mrr']['value'],
        'MRR CI low': result['metrics']['mrr']['ci_low'],
        'MRR CI high': result['metrics']['mrr']['ci_high'],
        'Recall@3': result['metrics']['recall_at_3']['value'],
        'false top-3 HHI': result['concentration']['false_top3_hhi'],
        'max false exposure': result['concentration']['maximum_false_top3_share'],
        'eligible': name in report['eligible_methods'],
    })
table = pd.DataFrame(rows).sort_values(['eligible', 'false top-3 HHI', 'MRR'], ascending=[False, True, False])
display(table)
exposure = pd.read_csv(OUT / 'hubness_reranking_author_exposure.csv')
tracked = exposure[exposure['profile'].str.contains('James Joyce|Katherine Mansfield|D. H. Lawrence', regex=True)]
display(tracked.sort_values(['method', 'false_top3_count'], ascending=[True, False]))
print('EXPLORATORY SELECTION:', report['selected_exploratory_method'])
print('RETURN:', OUT / 'hubness_reranking_metrics.json')
print('RETURN:', OUT / 'hubness_reranking_author_exposure.csv')